In [8]:
import os
import wandb
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
# VGG - Visual Geometry Group (The Group which created the model)

In [3]:
wandb.init(project="VGG-flower",config={
    "epochs":50,
    "batch_size": 16,
    "learning_rate":1e-3,
    "architecture":"VGG16",
    "pretrained":True,
    "input_size":(224,224)
})
config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ATRI SUKUL\_netrc.
wandb: Currently logged in as: atrisukul1508 (atrisukul1508-iittp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
config

{'epochs': 50, 'batch_size': 16, 'learning_rate': 0.001, 'architecture': 'VGG16', 'pretrained': True, 'input_size': [224, 224]}

In [5]:
data_transforms = {
    'train':transforms.Compose([
        transforms.Resize(config["input_size"]),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5],std=[0.5])
    ]),
    'val': transforms.Compose([
        transforms.Resize(config["input_size"]),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5],std=[0.5])
    ])
}

In [6]:
train_dir = './flowers/train'
val_dir = './flowers/val'

In [7]:
def get_dataset(datapath, transform):
    return datasets.ImageFolder(root=datapath, transform = transform)

In [14]:
def get_dataloader(dataset, batch_size, shuffle=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [10]:
model = models.vgg16(pretrained=config.pretrained)

for param in model.parameters():
    param.requires_grad_(False)

model.classifier[6] = nn.Linear(4096,5)

for param in model.classifier[6].parameters():
    param.requires_grad = True


c:\Users\ATRI SUKUL\deeplearning\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ATRI SUKUL\deeplearning\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [24]:
# model.classifier.append(nn.Linear(1000,10))

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[6].parameters(), lr= config['learning_rate'])

In [12]:
wandb.watch(model, criterion, log='all', log_freq=10, log_graph=True)

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


In [13]:
def train_model(model, criterion, optimizer, train_loader, val_loader, config):
    for epoch in range(config['epochs']):
        model.train()
        running_correct = 0
        running_total = 0
        batch_seen = 0
        for image, label in train_loader:
            optimizer.zero_grad()

            output = model(image)
            loss = criterion(output, label)
        
            loss.backward()

            optimizer.step()
            _, preds = torch.max(output, 1)
            
            batch_seen += 1
            running_correct += (preds == label).sum().item()
            running_total += len(label)


        running_acc = running_correct/running_total
    
        wandb.log({'loss': loss, 'epoch': epoch, 'acc':running_acc})
        print(f"Epoch: {epoch}, Epoch Acc: {running_acc}")

        model.eval()

        val_correct = 0
        val_total = 0

        for image, label in val_loader:
            outputs = model(image)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == label).sum().item()
            val_total += len(label)
        
        val_acc = val_correct/val_total
        wandb.log({'loss': loss, 'epoch': epoch, 'acc':val_acc})
        print(f"Epoch: {epoch}, Epoch Acc: {val_acc}")
        

In [15]:
train_ds = get_dataset(train_dir, data_transforms['train'])
val_ds = get_dataset(val_dir, data_transforms['val'])

In [16]:
train_loader = get_dataloader(train_ds, config['batch_size'], True)
val_loader = get_dataloader(val_ds, config['batch_size'])

In [ ]:
train_model(model, criterion, optimizer, train_loader, val_loader, config)